# Sparse Matrix-Vector Multiplication

## Operation Definition

Sparse matrix-vector multiplication, usually written SpMV, computes

$$
y = Ax.
$$

For CSR, each row gathers entries from `x`, multiplies by `data`, and reduces the row sum. SpMV is often limited by memory bandwidth rather than floating point throughput, a point emphasized in GPU-oriented SpMV studies {cite}`bell2009implementing`.

In [1]:
import time
import numpy as np
from scipy import sparse

In [2]:
rng = np.random.default_rng(7)
n = 50_000
entries_per_row = 10
density = entries_per_row / n

A = sparse.random(
    n,
    n,
    density=density,
    format="csr",
    random_state=rng,
    dtype=np.float64,
)
x = rng.standard_normal(n)

print("shape:", A.shape)
print("nnz:", A.nnz)
print("average nnz per row:", A.nnz / n)

shape: (50000, 50000)
nnz: 500000
average nnz per row: 10.0


## Basic Performance Model

A tiny benchmark is enough to see the basic performance model. One SpMV performs about `2 * nnz` floating point operations, but it also streams values, indices, row pointers, and vector entries.

In [3]:
def time_spmv(A, x, repeats=8):
    A @ x
    start = time.perf_counter()
    for _ in range(repeats):
        y = A @ x
    elapsed = (time.perf_counter() - start) / repeats
    return elapsed, y

elapsed, y = time_spmv(A, x)
gflop_s = (2 * A.nnz) / elapsed / 1e9

print(f"time per SpMV: {elapsed * 1e3:.3f} ms")
print(f"estimated rate: {gflop_s:.3f} GFLOP/s")
print("result norm:", np.linalg.norm(y))

time per SpMV: 0.422 ms
estimated rate: 2.369 GFLOP/s
result norm: 406.5922486209936


## Row Balance

Row length distribution matters for parallel SpMV. Balanced rows are easier to divide across threads or GPU warps; irregular rows cause load imbalance.

In [4]:
row_lengths = np.diff(A.indptr)
print("min row nnz:", row_lengths.min())
print("max row nnz:", row_lengths.max())
print("mean row nnz:", row_lengths.mean())
print("std row nnz:", row_lengths.std())

min row nnz: 0
max row nnz: 26
mean row nnz: 10.0
std row nnz: 3.154996038032378
